# 3주차. 나나의 기록장을 만들다

**부제:** 구조화된 출력을 SQLite에 저장: structured_requests, schedules, todos, reminders

## 0. 목표

이번 주에는 2주차 structured output을 실제 LangChain `create_agent(..., response_format=...)` 호출로 만들고, 그 결과를 SQLite row로 저장한다.

성취 기준:

- OpenAI API key를 사용하는 LangChain structured output 흐름을 실행할 수 있다.
- `structured_requests`, `schedules`, `todos`, `reminders` table의 역할을 구분할 수 있다.
- `kind`에 따라 알맞은 table에 정규화 저장되는지 trace로 확인할 수 있다.

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

3주차도 실제 OpenAI API를 호출한다. `.env`의 `OPENAI_API_KEY`, `OPENAI_MODEL`이 설정되어 있어야 한다.

In [1]:
from __future__ import annotations

import json
import os
import sqlite3
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env", override=True)
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("repo 루트 .env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 700) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace: list[dict[str, Any]] = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace

DB_PATH = Path(os.getenv("KANAMATE_WEEK03_DB_PATH") or REPO_ROOT / "tmp" / "week03_structured_requests.sqlite3").resolve()
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
show_json({"model": OPENAI_MODEL, "db_path": str(DB_PATH)})

{
  "model": "gpt-4o-mini",
  "db_path": "/Users/ssojux2/Working/kakao_clone_coding/tmp/week03_structured_requests.sqlite3"
}


## 2. 개념

오늘의 큰 그림: 모델은 자연어 요청을 Pydantic payload로 구조화하고, 앱은 그 payload를 SQLite에 저장한다. 모든 원본 structured output은 `structured_requests`에 저장하고, 일정/할 일/알림은 각각의 table에 정규화한다.

## 3. 기본 개념 실습

먼저 Pydantic schema와 LangChain structured output agent를 만든다.

In [2]:
class StructuredRequest(BaseModel):
    kind: Literal["personal_schedule", "group_schedule", "todo", "reminder", "unknown"] = Field(description="사용자 요청 종류")
    title: str | None = Field(default=None, description="일정/할 일/알림 제목")
    date: str | None = Field(default=None, description="YYYY-MM-DD 날짜")
    start_time: str | None = Field(default=None, description="HH:MM 시작 시각")
    end_time: str | None = Field(default=None, description="HH:MM 종료 시각")
    members: list[str] = Field(default_factory=list, description="관련 참석자")
    priority: str | None = Field(default=None, description="low, medium, high 중 하나")
    reason: str | None = Field(default=None, description="분류 또는 추출 근거")


structured_agent = create_agent(
    model=make_model(700),
    tools=[],
    response_format=StructuredRequest,
    system_prompt=(
        "너는 카나메이트 요청 추출기다. 오늘은 2026-05-15이다. "
        "사용자 요청을 personal_schedule, group_schedule, todo, reminder, unknown 중 하나로 분류하고 "
        "title/date/start_time/end_time/members/priority/reason을 검증 가능한 값으로 채운다."
    ),
)

request_text = "민수 지아랑 다음 주 화요일 3시에 회의 잡아줘. 중요한 일정이야."
structured_result = structured_agent.invoke({"messages": [{"role": "user", "content": request_text}]})
structured_payload = structured_result["structured_response"].model_dump()
show_json(structured_payload)
assert structured_payload["kind"] in {"personal_schedule", "group_schedule", "todo", "reminder", "unknown"}

{
  "kind": "group_schedule",
  "title": "회의",
  "date": "2026-05-19",
  "start_time": "15:00",
  "end_time": "16:00",
  "members": [
    "민수",
    "지아"
  ],
  "priority": "high",
  "reason": "중요한 일정"
}


## 4. 카나메이트 확장 예제

SQLite schema를 만들고, structured output payload를 `kind`에 맞는 table로 정규화 저장한다.

In [3]:
def connect_db() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def initialize_structured_db(reset: bool = False) -> None:
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS structured_requests (
                request_id TEXT PRIMARY KEY,
                source_text TEXT NOT NULL,
                kind TEXT NOT NULL,
                payload_json TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS schedules (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                request_id TEXT NOT NULL,
                title TEXT NOT NULL,
                date TEXT,
                start_time TEXT,
                end_time TEXT,
                members_json TEXT NOT NULL,
                reason TEXT,
                schedule_type TEXT NOT NULL,
                FOREIGN KEY (request_id) REFERENCES structured_requests(request_id)
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS todos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                request_id TEXT NOT NULL,
                title TEXT NOT NULL,
                due_date TEXT,
                priority TEXT,
                reason TEXT,
                FOREIGN KEY (request_id) REFERENCES structured_requests(request_id)
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS reminders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                request_id TEXT NOT NULL,
                title TEXT NOT NULL,
                date TEXT,
                start_time TEXT,
                reason TEXT,
                FOREIGN KEY (request_id) REFERENCES structured_requests(request_id)
            )
        """)
        if reset:
            conn.execute("DELETE FROM reminders")
            conn.execute("DELETE FROM todos")
            conn.execute("DELETE FROM schedules")
            conn.execute("DELETE FROM structured_requests")
        conn.commit()


def save_structured_output(source_text: str, payload: dict[str, Any]) -> dict[str, Any]:
    request_id = f"req-{uuid.uuid4().hex[:12]}"
    kind = payload["kind"]
    title = payload.get("title") or "제목 없음"
    with connect_db() as conn:
        conn.execute(
            "INSERT INTO structured_requests (request_id, source_text, kind, payload_json, created_at) VALUES (?, ?, ?, ?, ?)",
            (request_id, source_text, kind, json.dumps(payload, ensure_ascii=False), now_iso()),
        )
        if kind in {"personal_schedule", "group_schedule"}:
            conn.execute(
                """INSERT INTO schedules
                   (request_id, title, date, start_time, end_time, members_json, reason, schedule_type)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                (request_id, title, payload.get("date"), payload.get("start_time"), payload.get("end_time"), json.dumps(payload.get("members", []), ensure_ascii=False), payload.get("reason"), kind),
            )
        elif kind == "todo":
            conn.execute(
                "INSERT INTO todos (request_id, title, due_date, priority, reason) VALUES (?, ?, ?, ?, ?)",
                (request_id, title, payload.get("date"), payload.get("priority"), payload.get("reason")),
            )
        elif kind == "reminder":
            conn.execute(
                "INSERT INTO reminders (request_id, title, date, start_time, reason) VALUES (?, ?, ?, ?, ?)",
                (request_id, title, payload.get("date"), payload.get("start_time"), payload.get("reason")),
            )
        conn.commit()
    return {"request_id": request_id, "kind": kind}

initialize_structured_db(reset=True)
save_trace = save_structured_output(request_text, structured_payload)
show_json(save_trace)

{
  "request_id": "req-b8710aee64eb",
  "kind": "group_schedule"
}


## 5. 확장 예제 실행

저장 후에는 원본 structured request와 정규화 table을 따로 조회한다. UI 좌측 대화 목록과 별개로 추출된 일정/할 일/알림 row가 남는지 확인한다.

In [4]:
def fetch_all(table: str) -> list[dict[str, Any]]:
    with connect_db() as conn:
        rows = conn.execute(f"SELECT * FROM {table} ORDER BY rowid").fetchall()
    return [dict(row) for row in rows]

trace = {
    "structured_requests": fetch_all("structured_requests"),
    "schedules": fetch_all("schedules"),
    "todos": fetch_all("todos"),
    "reminders": fetch_all("reminders"),
}

assert len(trace["structured_requests"]) == 1
assert sum(len(trace[name]) for name in ["schedules", "todos", "reminders"]) in {0, 1}
show_json(trace)

{
  "structured_requests": [
    {
      "request_id": "req-b8710aee64eb",
      "source_text": "민수 지아랑 다음 주 화요일 3시에 회의 잡아줘. 중요한 일정이야.",
      "kind": "group_schedule",
      "payload_json": "{\"kind\": \"group_schedule\", \"title\": \"회의\", \"date\": \"2026-05-19\", \"start_time\": \"15:00\", \"end_time\": \"16:00\", \"members\": [\"민수\", \"지아\"], \"priority\": \"high\", \"reason\": \"중요한 일정\"}",
      "created_at": "2026-05-15T09:15:07+00:00"
    }
  ],
  "schedules": [
    {
      "id": 4,
      "request_id": "req-b8710aee64eb",
      "title": "회의",
      "date": "2026-05-19",
      "start_time": "15:00",
      "end_time": "16:00",
      "members_json": "[\"민수\", \"지아\"]",
      "reason": "중요한 일정",
      "schedule_type": "group_schedule"
    }
  ],
  "todos": [],
  "reminders": []
}


## 6. 회고

확인 질문:

1. LangChain structured output을 바로 DB에 저장할 때 어떤 필드를 검증해야 하는가?
2. 원본 payload를 `structured_requests`에 남기는 이유는 무엇인가?
3. `personal_schedule`과 `group_schedule`을 같은 `schedules` table에 저장하면서도 구분하려면 어떤 컬럼이 필요한가?

작은 응용 과제: `todo`, `reminder`, `unknown` 요청을 각각 실행하고 저장 row가 어떻게 달라지는지 비교한다.